In [ ]:
import json
import os
from time import time

from dotenv import load_dotenv
from azure.cosmos import CosmosClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
import requests
from pathlib import Path
from openai import AzureOpenAI
from Backend.models import Product, ProductList

import json
from pydantic import BaseModel, ValidationError
from typing import Type, TypeVar, List
from azure.cosmos import CosmosClient, DatabaseProxy, ContainerProxy, PartitionKey
import time
#from tenacity import retry, wait_random_exponential, stop_after_attempt

loaded = load_dotenv()
if not loaded:
    raise RuntimeError(".env file could not be loaded.")

key_vault_url = os.environ["KEY_VAULT_URL"]
secret_name = os.environ["COSMOS_DB_CONNECTION_STRING_SECRET_NAME"]

credential = DefaultAzureCredential()
secret_client = SecretClient(vault_url=key_vault_url, credential=credential)
connection_string = secret_client.get_secret(secret_name).value

if connection_string is None:
    raise RuntimeError(f"Key Vault secret '{secret_name}' has no value.")

client = CosmosClient.from_connection_string(connection_string)

DatabaseProxy = client.create_database_if_not_exists("test_db_5")

ContainerProxy = DatabaseProxy.create_container_if_not_exists(
    id="product",
    partition_key={"paths": ["/categoryId"], "kind": "Hash"}
)

# Create a document
# ContainerProxy.upsert_item({"id": "1", "categoryId": "electronics"})

# items = ContainerProxy.query_items(query="SELECT * FROM product", enable_cross_partition_query=True)

# Add product data to database using batch operations
# Get cosmic works product data from github
# product_raw_data = "https://cosmosdbcosmicworks.blob.core.windows.net/cosmic-works-small/product.json"
# product_data = ProductList(items=[Product(**data) for data in requests.get(product_raw_data).json()])
# for prod in product_data.items:
#     print(prod)
#     ContainerProxy.upsert_item(prod.model_dump(by_alias=True))

# product_raw_data = "https://cosmosdbcosmicworks.blob.core.windows.net/cosmic-works-small/product.json"
# product_data = ProductList(items=[Product(**data) for data in requests.get(product_raw_data).json()])
# for prod in product_data.items:
#     ContainerProxy.upsert_item(prod.model_dump(by_alias=True))

# product_file = Path("products.json")
# with product_file.open(encoding="utf-8") as file:
#     product_data = ProductList(
#         items=[Product(**data) for data in json.load(file)]
#     )

# for prod in product_data.items:
#     ContainerProxy.upsert_item(prod.model_dump(by_alias=True))

aoai_endpoint = os.environ["AOAI_ENDPOINT"]
api_version = os.environ["AOAI_API_VERSION"]
aoai_key = os.environ["AOAI_KEY"]

# Instantiate an AzureOpenAI client
ai_client = AzureOpenAI(
    azure_endpoint = aoai_endpoint,
    api_version = api_version,
    api_key = aoai_key
    )

# Generate embeddings from a text string
embeddings_deployment_name = os.environ["EMBEDDINGS_DEPLOYMENT_NAME"]

def generate_embeddings(text: str):
    '''
    Generate embeddings from string of text using the deployed Azure OpenAI API embeddings model.
    This will be used to vectorize document data and incoming user messages for a similarity search with
    the vector index.
    '''
    response = ai_client.embeddings.create(input=text, model=embeddings_deployment_name)
    embeddings = response.data[0].embedding
    time.sleep(0.5) # rest period to avoid rate limiting on AOAI
    return embeddings

# Create the vector embedding policy
vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/contentVector",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 1536
        }
    ]
}

# Create the indexing policy
indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [
        {
            "path": "/*"
        }
    ],
    "excludedPaths": [
        {
            "path": "/\"_etag\"/?"
        },
        {
            "path": "/contentVector/*"
        }
    ],
    "vectorIndexes": [
        {
            "path": "/contentVector",
            "type": "diskANN"
        }
    ]
}

product_v_container = DatabaseProxy.create_container_if_not_exists(
    id="product_v",
    partition_key=PartitionKey(path="/categoryId"),
    indexing_policy=indexing_policy,
    vector_embedding_policy=vector_embedding_policy
)

### Performing a vector search query
def vector_search(
        container: ContainerProxy, 
        prompt: str,         
        vector_field_name:str="contentVector", 
        num_results:int=5):
    query_embedding = generate_embeddings(prompt)    
    items = container.query_items(
        query=f"""SELECT TOP @num_results itm.id, VectorDistance(itm.{vector_field_name}, @embedding) AS SimilarityScore 
                FROM itm
                ORDER BY VectorDistance(itm.{vector_field_name}, @embedding)
                """,
        parameters = [
            { "name": "@num_results", "value": num_results },
            { "name": "@embedding", "value": query_embedding }            
        ],
        enable_cross_partition_query=True
        )
    return items

### Using vector search results with an LLM in a RAG scenario
def rag_with_vector_search(
        container: ContainerProxy, 
        prompt: str,         
        vector_field_name:str="contentVector", 
        num_results:int=5):
    """
    Use the RAG model to generate a prompt using vector search results based on the
    incoming question.
    """
    # perform the vector search and build product list
    results = vector_search(container, prompt, vector_field_name, num_results)
    product_list = ""
    for result in results:
        # retrieve the product details
        product = query_item_by_id(container, result["id"], Product)               
        # remove the contentVector field from the product details, this isn't needed for the context
        product.content_vector = None        
        product_list += json.dumps(product, indent=4, default=str) + "\n\n"

    # generate prompt for the LLM using vector search results
    formatted_prompt = system_prompt + product_list

    # prepare the LLM request
    messages = [
        {"role": "system", "content": formatted_prompt},
        {"role": "user", "content": prompt}
    ]

    completion = ai_client.chat.completions.create(messages=messages, model=deployment_name)
    return completion.choices[0].message.content